# Выборочное исследование данных

## Упрощение чтения

Сами по себе json'ы очень сложные для простого человеческого чтения.
Структуры очень вложенные, но хотелось бы понять, какие top level атрибуты есть у каждой сущности.

Сформируем набор файлов с постфиксом `_keys`, для того, чтобы иметь представление о том, какие атрибуты есть у каждой сущности.

In [1]:
# imports
import os
import json
import json_utils
from pathlib import Path

In [2]:
def prepare_for_serialization(value: dict) -> dict:
    serialized_data = {}

    for key, value in value.items():
        if isinstance(value, set):
            class_obj = next(iter(value))
            type_name = class_obj.__name__  # extract just 'int', 'str', etc.
            serialized_data[key] = type_name
    
    return serialized_data

def generate_compact_entity_preview(input_path: str, artifacts_path: str):
    os.makedirs(artifacts_path, exist_ok=True)
    # Get all files in folder
    for filename in os.listdir(input_path):
        file_path = os.path.join(input_path, filename)
        
        # Check if it's a file (not a subfolder)
        if os.path.isfile(file_path):
            entity_top_level_keys = json_utils.describe_entity_from_json(file_path)
            entity_keys_serialized = prepare_for_serialization(entity_top_level_keys)

            # Save to new file
            output_filename = f"{os.path.splitext(filename)[0]}_keys.json"
            output_path = os.path.join(artifacts_path, output_filename)
            
            with open(output_path, 'w') as f:
                f.write(json.dumps(entity_keys_serialized, indent=2))
            
            print(f"Processed: {filename} -> {output_filename}")

In [3]:
# define path constants
INPUT_DATA_PATH = '../../data'
ARTIFACTS_DATA_PATH = '../artifacts/structures'

In [4]:
generate_compact_entity_preview(input_path=INPUT_DATA_PATH, artifacts_path=ARTIFACTS_DATA_PATH)

Processed: datasets.json -> datasets_keys.json
Processed: press-media.json -> press-media_keys.json
Processed: courses.json -> courses_keys.json
Processed: external-organisations.json -> external-organisations_keys.json
Processed: award-milestones.json -> award-milestones_keys.json
Processed: funding-opportunities.json -> funding-opportunities_keys.json
Processed: author-collaborations.json -> author-collaborations_keys.json
Processed: equipments.json -> equipments_keys.json
Processed: curricula-vitae.json -> curricula-vitae_keys.json
Processed: external-persons.json -> external-persons_keys.json
Processed: impacts.json -> impacts_keys.json
Processed: classification-schemes.json -> classification-schemes_keys.json
Processed: journals.json -> journals_keys.json
Processed: prizes.json -> prizes_keys.json
Processed: awards.json -> awards_keys.json
Processed: semantic-groups.json -> semantic-groups_keys.json
Processed: fingerprints.json -> fingerprints_keys.json
Processed: activities.json 

## Постановка задачи

Сущностей очень много.
Нужно попытаться понять, можем ли мы на таком наборе данных соединять разные сущности между собой.

Анализируя вручную, было принято решение посмотреть на связку между `persons` и `organisational-units`.

### Организации

#### Подготовка данных

Начнем работать с ними, т.к. у организации меньше полей, с которыми надо разобраться.

Согласно [документации Pure](https://helpcenter.pure.elsevier.com/organisational-unit):

> Organisational units are a research institution’s schools, faculties, institutes, departments, and so on; any type of unit that makes up an organisation.
> If these units are organised hierarchically, you can model the structure in Pure.

In [5]:
# load items from json
organizations = json_utils.load_from_json(os.path.join(INPUT_DATA_PATH, 'organisational-units.json'))
organizations[0]

{'pureId': 19926,
 'externalId': '50000001',
 'externalIdSource': 'synchronisedUnifiedOrganisation',
 'externallyManaged': True,
 'uuid': 'abf8fae5-478f-4b63-8a8c-944750655c44',
 'period': {'startDate': '1900-01-01T12:00:00.000+02:30'},
 'info': {'createdBy': 'sync_user',
  'createdDate': '2017-05-11T15:49:21.344+0300',
  'modifiedBy': 'sync_user',
  'modifiedDate': '2025-03-01T11:28:34.778+0300',
  'portalUrl': 'https://pureportal.spbu.ru/en/organisations/federal-state-budgetary-educational-institution-of-higher-educationsaint-petersburg-state-university(abf8fae5-478f-4b63-8a8c-944750655c44).html',
  'prettyURLIdentifiers': ['федеральное-государственное-бюджетное-образовательное-учреждение-',
   'federal-state-budgetary-educational-institution-of-higher-educati']},
 'name': {'formatted': False,
  'text': [{'locale': 'en_US',
    'value': 'Federal State Budgetary Educational Institution of Higher Education"Saint Petersburg State University"'},
   {'locale': 'ru_RU',
    'value': 'Федер

Смотря на результаты `organisational-units_keys.json` и на данные в `organisational-units.json`, можно проигнорировать некоторые поля на данный момент.

Интересуют следующие поля - `uuid`, `level`, `parents`.

`uuid` - уникальный идентификатор сущности.
По всей видимости, можно вытаскивать связи между сущностями при помощи `uuid`, а не по `pureId`.

Про это написано в [документации Pure](https://helpcenter.pure.elsevier.com/understanding-dependents-in-the-api):

> - UUIDs are always present in content and serve as unique identifiers.
> - The API does not use traditional database constraints like primary and foreign keys.
> - Instead, relationships are retrieved dynamically through the DEPENDENTS endpoints, which identify content that relies on a given record.
> This helps you understand dependencies between content types and how they are connected. 
> This allows you to create logical associations.

`level`, по всей видимости, указывает на место в иерархии.

`parents`, по всей видимости, содержит `uuid` родительских отделений.

In [6]:
def extract_organization_data(organizations: list) -> list:
    # Return a list of organizational units (a more compact one)
    # basically we reduce the amount of fields we want to explore

    result = list()

    for organization in organizations:
        organization_unit = dict()

        organization_unit['uuid'] = organization['uuid'] # unit uuid

        # organization_unit['name'] = organization['name']['text'][0]['value'] # unit name (ru locale)
        # CAUTION: works if we are sure that there will always be ru_RU locale
        organization_names = organization['name']['text']
        for name in organization_names:
            if name['locale'] == 'ru_RU':
                organization_unit['name'] = name['value']
        
        organization_unit['level'] = organization['type']['term']['text'][0]['value'] # unit level

        # if we have parents, should add them:
        if 'parents' in organization.keys():
            parents = list()
            for parent in organization['parents']:
                parents.append(parent['uuid'])
            organization_unit['parents'] = parents
        
        result.append(organization_unit)
    
    return result

Т.к. мы сжимаем количество данных до формата, который удобно представить в виде читаемой в Jupyter Notebook структуры, подключим `pandas` для построения датафрейма.

Будем использовать это как in-memory альтернативу для построения запросов к нашей "базе данных".

In [7]:
# imports
import pandas as pd

In [8]:
organizations_reduced = extract_organization_data(organizations)
organization_df = pd.DataFrame(organizations_reduced)

In [9]:
# use to see column types and memory usage
organization_df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   uuid     1000 non-null   str   
 1   name     1000 non-null   str   
 2   level    1000 non-null   str   
 3   parents  999 non-null    object
dtypes: object(1), str(3)
memory usage: 31.4+ KB


In [10]:
# take random sample of 7 elements too see the data
organization_df.sample(7)

,uuid,name,level,parents
581,8616cfdd-c1f9-4df6-ad97-635fec0a6111,Кафедра русского языка,Level 2,[26619258-c0ee-48ed-a39e-bed170df5579]
52,ec2d9f55-eec1-42ac-980a-01a82a37ace0,MK.2539 Конституционное право; конституционный...,Level 2,"[c8cd6768-8255-439c-8bf8-eb6966fa0924, 48d3663..."
425,84ce3ff9-7ede-41ad-a80d-568d8cc0d4fa,Управление по организации питания в СПбГУ,Level 2,[e416cd64-6d13-4bdf-97e7-d4d7ab961026]
858,29126e96-d385-48a0-bd05-1bc9933eeca3,MK.3052.2014 История искусства,Level 3,[911670ec-5d23-42af-a1d5-3dc2d737a99e]
857,e5aa8421-c43d-4d1b-b9de-5ee8d911b6ed,"MK.3051.2016 Философия, этика и религиоведение",Level 3,[7d1a2fbc-98d9-4f3b-9a60-43aa99a8ad6d]
137,f1c7ea0a-4525-42b3-9c16-b160eb99ab96,MK.3043 Литература народов стран Азии и Африки,Level 2,[48d3663f-c069-4000-bb53-efeec4d313b2]
797,9e818394-a2b9-425e-b9f7-3f66a6317ce2,MK.3026.2016 Экономика,Level 3,[160a6e95-c3a5-461c-9853-036fbc7852b1]


Посмотрим, за что отвечает `level`.

Из каждой группы `level` вытащим по 3 записи, чтобы почитать их названия.

In [11]:
# show 3 organizational units per level
organization_df.groupby('level').head(3)

,uuid,name,level,parents
0,abf8fae5-478f-4b63-8a8c-944750655c44,Федеральное государственное бюджетное образова...,Level 0,NaN
1,48d3663f-c069-4000-bb53-efeec4d313b2,аспирантура,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
2,65666392-9044-41de-aebd-3f58d14f5679,ординатура,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
3,436337a2-0866-4388-9baf-340bd3a55aae,Институт наук о Земле СПбГУ,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
39,32fa2285-42f1-42c0-8ed1-7ef03a75fe77,"MK.2503 Теория, методология и история социологии",Level 2,[48d3663f-c069-4000-bb53-efeec4d313b2]
40,754dba6e-4686-46e2-baa0-adb0bba2edd1,MK.2506 Экономическая теория,Level 2,[48d3663f-c069-4000-bb53-efeec4d313b2]
41,49250bba-7f2e-45b3-8f41-41f46edcd5de,"MK.2508 Финансы, денежное обращение и кредит",Level 2,[48d3663f-c069-4000-bb53-efeec4d313b2]
417,4727c0db-5596-4e97-adb7-30e54c90a3b8,Учебное управление,Level 3,[4891599d-ebbb-465a-a1ba-5e4cb5f1cf24]
426,56fbb52a-3ddb-4469-a7ab-7db6abe717b6,Управление образовательных программ,Level 3,[4891599d-ebbb-465a-a1ba-5e4cb5f1cf24]
644,b0dab385-587f-46ec-b8ae-af46f53a1b95,"MK.2503.2010 Теория, методология и история соц...",Level 3,[32fa2285-42f1-42c0-8ed1-7ef03a75fe77]


`Level 0` есть только у одной сущности из присланной выборки.
И это сам СПБГУ.

Выборочно посмотрим на каждые из `level`, которые у нас есть в выборке.

In [12]:
filtered_by_level = organization_df[organization_df['level'] == 'Level 1']
filtered_by_level.head(10)

,uuid,name,level,parents
1,48d3663f-c069-4000-bb53-efeec4d313b2,аспирантура,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
2,65666392-9044-41de-aebd-3f58d14f5679,ординатура,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
3,436337a2-0866-4388-9baf-340bd3a55aae,Институт наук о Земле СПбГУ,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
4,823e3ac7-e97a-42ce-8678-59b721259afb,"Специализированный учебно-научный центр ""Акаде...",Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
5,d6fddcd6-444c-4113-938a-facd8c032b6e,Биологический факультет,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
6,b239ccdd-36c9-4f7a-8c0f-63a973167aa6,Восточный факультет,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
7,84b6b9ba-35db-44c2-8ca2-6f50edc771c3,Высшая школа менеджмента,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
8,1249292b-434e-4ea8-9d38-879f8ae406ef,"Институт ""Высшая школа журналистики и массовых...",Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
9,4c45cb39-1632-49a0-b4a6-44b6d08d49da,Институт химии Санкт-Петербургского государств...,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]
10,dc1e6e64-b14c-4129-91d0-71aa5168da69,Институт философии СПбГУ,Level 1,[abf8fae5-478f-4b63-8a8c-944750655c44]


Что можно сказать про последующие уровни?

`level 1` - факультеты и аспирантура / ординатура (почему?), их `parent` - спбгу

`level 2` - кафедры и общие названия программ (?)

`level 3` - какие-то сектора и более детализированные образовательные программы

**Скорее всего тут можно выстроить иерархию, древовидную структуру с корнем в спбгу!**

#### Выбор более узкой задачи

Поскольку у нас есть ПМ-ПУ в выгрузке, можем задаться более конкретной задачей: получить все организации, которые связаны с пм-пу в данной выборке.

In [13]:
apmath_uuid = '0435d70c-2eef-4944-90ed-649c9118ccac'
apmath_units = organization_df[
        organization_df['parents']
        .apply(
            lambda x: isinstance(x, list) and apmath_uuid in x
        )
    ]
apmath_units

,uuid,name,level,parents
447,3343d1c4-5fda-4eeb-b8b7-99b4d0486dd7,Кафедра моделирования электромеханических и ко...,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]
448,11caa15b-33da-4386-a6f7-29f247d59ede,Кафедра высшей математики,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]
449,f5159fdf-72d8-4c07-ac04-a2577e85535b,Кафедра вычислительных методов механики деформ...,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]
450,df048f85-9f98-44d0-ab7f-8c6ca7ba6481,Кафедра диагностики функциональных систем,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]
451,1a947b4c-6f8c-42ef-b3d8-9e20c3c29ed8,Кафедра информационных систем,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]
452,18e307e1-04a2-46eb-a50e-9975b0c7d452,Кафедра компьютерного моделирования и многопро...,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]
453,786faf0d-4b5e-41b1-aa8e-1c7da0ac1d80,Кафедра компьютерных технологий и систем,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]
454,1fffa552-7fe9-4e35-96d4-12ac1cfd03a9,Кафедра космических технологий и прикладной ас...,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]
455,3672f23a-61a7-4189-8869-6384cc2ee273,Кафедра математической теории микропроцессорны...,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]
456,5d649981-51f1-4792-b407-cf01c50f5c8d,Кафедра математической теории моделирования си...,Level 2,[0435d70c-2eef-4944-90ed-649c9118ccac]


In [14]:
# save apmath units uuids to use them with persons
apmath_units_uuid = apmath_units['uuid']
apmath_units_uuid_list = apmath_units_uuid.tolist()

### Люди

#### Подготовка данных

Согласно [документации Pure](https://helpcenter.pure.elsevier.com/take-advantage-of-the-person-profile):

> Contains a researcher's name, name variants, titles, IDs, links and more.

In [15]:
# load items from json
persons = json_utils.load_from_json(os.path.join(INPUT_DATA_PATH, 'persons.json'))
persons[0]

{'pureId': 143835,
 'externalId': '50063896',
 'externalIdSource': 'synchronisedUnifiedPerson',
 'uuid': '2b5c936a-4af4-44f9-9cc3-4e47a2cf4ba2',
 'name': {'firstName': 'Евгений Александрович', 'lastName': 'Поляков'},
 'orcid': '0000-0001-9850-5370',
 'fte': 0.0,
 'isExpert': False,
 'info': {'createdBy': 'sync_user',
  'createdDate': '2017-05-11T19:40:46.645+0300',
  'modifiedBy': 'root',
  'modifiedDate': '2018-06-09T04:50:41.314+0300',
  'portalUrl': 'https://pureportal.spbu.ru/en/persons/--(2b5c936a-4af4-44f9-9cc3-4e47a2cf4ba2).html',
  'prettyURLIdentifiers': ['евгений-александрович-поляков']},
 'visibility': {'key': 'BACKEND',
  'value': {'formatted': False,
   'text': [{'locale': 'en_US', 'value': 'Backend - Restricted to Pure users'},
    {'locale': 'ru_RU',
     'value': 'Сервер - доступно только пользователям Pure'}]}},
 'nameVariants': [{'pureId': 11328091,
   'externalId': '50063896',
   'externalIdSource': 'synchronisedUnifiedPerson',
   'name': {'firstName': 'Evgenii', 'la

Согласно документации Pure, имя человека можно выделить из объекта `name`.

Посмотрим на структуру объекта для первого человека из выборки

In [16]:
# take name
persons[0]['name']

{'firstName': 'Евгений Александрович', 'lastName': 'Поляков'}

Хотелось бы узнать, как связаны между собой человек и организация.

Видимо, есть 2 основных типа организаций, к которым может принадлежать человек - `staffOrganisationAssociations` и `studentOrganisationAssociations`.
Предположительно, первый атрибут обозначает принадлежность человека к организации как сотрудника.

Это подтверждается [спецификацией Pure](https://api.elsevierpure.com/ws/api/api-docs/index.html?url=/ws/api/openapi.yaml#/person/person_get):

> Organizations that the person is associated with as 'Staff'

Исходя из результатов в `persons_keys.json`, таких организаций у одного пользователя может быть несколько.
Посмотрим, как это можно вызвать из кода:

In [17]:
# can take uuid of organisation
persons[0]['staffOrganisationAssociations'][0]['organisationalUnit']

{'uuid': '3c70bc8d-1758-41dd-b716-5c21d94bd23f',
 'link': {'ref': 'content',
  'href': 'http://localhost:8080/ws/api/522/organisational-units/3c70bc8d-1758-41dd-b716-5c21d94bd23f'},
 'externalId': '50116349',
 'externalIdSource': 'synchronisedUnifiedOrganisation',
 'externallyManaged': True,
 'name': {'formatted': False,
  'text': [{'locale': 'en_US',
    'value': 'Department of Molecular Biophysics and Polymer Physics'},
   {'locale': 'ru_RU',
    'value': 'Кафедра молекулярной биофизики и физики полимеров'}]},
 'type': {'pureId': 17281,
  'uri': '/dk/atira/pure/organisation/organisationtypes/organisation/level_2',
  'term': {'formatted': False,
   'text': [{'locale': 'en_US', 'value': 'Level 2'},
    {'locale': 'ru_RU', 'value': '2 уровень'}]}}}

Получается, `person` содержит в себе довольно подробную информацию об организации, а не только ее `uuid`.

Опять же, все поля сейчас просматривать не имеет смысла.
Ограничимся `uuid`, `name` и `uuid` организаций, с которыми связан пользователь.

In [18]:
def extract_person_data(persons: list) -> list:
    # i want to take person uuid, name, organizational unit
    result = list()

    for person in persons:
        person_unit = dict()

        person_unit['uuid'] = person['uuid'] # unit uuid
        person_unit['name'] = f"{person['name']['lastName']} {person['name']['firstName']}"
        
        person_organizations = person['staffOrganisationAssociations']
        organization_uuids = list()
        organization_jobs = list()

        for organization in person_organizations:
            organization_uuids.append(organization['organisationalUnit']['uuid'])

            # CAUTION: doesn't work since jobTitle is not a required field
            # job_title_names = organization['jobTitle']['term']['text']
            # print(job_title_names)
            # for job in job_title_names:
            #     if job['locale'] == 'ru_RU':
            #         organization_jobs.append(job['value'])

        person_unit['organizations'] = organization_uuids
        result.append(person_unit)
    
    return result

Сформируем pandas dataframe для дальнейшей работы

In [19]:
persons_reduced = extract_person_data(persons)
person_df = pd.DataFrame(persons_reduced)

In [20]:
# get random sample of persons
person_df.sample(7)

,uuid,name,organizations
201,c19d7f4d-5ed4-45a6-8a9c-4c77f6d787bc,Аствацатурова Вера Викторовна,[4d26fb1b-2405-473c-8b86-78087d16318e]
483,55102b4e-2aa2-420a-9e87-6ed215eaadea,Беликова Виолетта Сергеевна,[8417de89-b12b-4d9e-927b-e197aa61e507]
569,be4675ba-a426-4c36-aa10-ff50df13c4d6,Кербунова Ирина Юрьевна,[020e835c-b2ba-455c-aa07-9152baf09c57]
149,e5700f26-0213-498f-aa78-7c6a9f64133e,Товстик Татьяна Михайловна,"[12efada2-270e-42c3-aa8e-d30bd61b591f, 12efada..."
21,54d0b3d4-5222-4fb6-b4e0-7d9e9ed0aeb8,Петрунькин Алексей Михайлович,[010792f1-f845-470f-b692-e25a90695e17]
48,ecd03e53-339a-4e6f-a331-ab982ca377a3,Козырева Нэлли Владимировна,"[f6b58bb8-7deb-432d-bab2-e2835f5bba0d, 1ea11a0..."
349,29f9b815-c3d5-41a5-b41a-1912796a8cc3,Куликова Елена Леонидовна,"[fd86dcb7-7292-4dd1-a7a7-a1a385a57f39, fd86dcb..."


#### Выбор более узкой задачи

Ранее мы уже нашли `uuid`s, которые относятся к подразделениям пм-пу.
Теперь посмотрим, есть ли в нашей выборке сотрудники из этих подразделений.

In [21]:
apmath_persons = person_df[
        person_df['organizations']
        .apply(
            lambda x: isinstance(x, list) and any(uuid in x for uuid in apmath_units_uuid_list)
        )
    ]
apmath_persons

,uuid,name,organizations
159,b11c249c-18ee-4494-a897-3966da97064d,Балыкина Юлия Ефимовна,"[813a4900-f5da-4667-bead-59a224b4bcd2, 813a490..."
163,fa04ec63-d948-435a-aaff-fbc46255ec84,Александрова Ирина Васильевна,"[489d6770-fff8-41b1-be91-34f1002e9c22, 489d677..."
204,699e6dfa-251b-44f9-8599-8c99ff069e52,Смирнов Николай Васильевич,"[652ed6df-9d33-409c-add5-1406d63ef348, 652ed6d..."
205,1a34decc-5009-4cdb-a37f-f59eadc44494,Кузютин Денис Вячеславович,"[bf6043ae-7ecb-4032-bfc7-1a9c056d5766, bf6043a..."
223,250c452b-d714-44d5-bccb-c44e077aac77,Парфенов Андрей Павлович,"[df048f85-9f98-44d0-ab7f-8c6ca7ba6481, df048f8..."
243,b4146396-6229-4b37-92f3-433b8485e875,Никитин Александр Владимирович,[3672f23a-61a7-4189-8869-6384cc2ee273]
255,a63ef843-db4e-4738-a720-07fc04e27875,Чашников Михаил Викторович,"[489d6770-fff8-41b1-be91-34f1002e9c22, 489d677..."
353,2f974cbd-2f8d-4b16-b690-277feb6a2f52,Зенкевич Николай Анатольевич,"[d97a9c77-1d9c-44c8-85c7-ad928ff3182c, d97a9c7..."
385,14bbd794-9b4a-46cc-8820-529398a81c83,Курбатова Галина Ибрагимовна,"[3343d1c4-5fda-4eeb-b8b7-99b4d0486dd7, 3343d1c..."
403,bdf2ff2d-f1fc-435f-ad99-70707041cf7a,Зубов Афанасий Владимирович,[3672f23a-61a7-4189-8869-6384cc2ee273]


## Выводы

### Ценность текущих результатов

Как видно, на полученной выборке можно получать агрегированные данные.

Структура сущностей получается довольно громоздкой, потому что при наличии связей с другими сущностями включается не только связка в виде `uuid`, но и более подробная информация, которая затрудняет быстрый разбор и интерпретацию атрибутов, особенно при отсутствии документации.

В спецификации, представленной в вики репозитория, отсутствует описание ответов с сервера, поэтому изначально пришлось выстраивать предположения о том, что значит тот или иной атрибут, исходя из информации на Pure Helpdesk.

Лишь в конце исследования удалось найти общую спецификацию к Pure, которая довольно подробно описывает ответы в Pure API - [ссылка](https://api.elsevierpure.com/ws/api/api-docs/index.html?url=/ws/api/openapi.yaml).

**Поскольку это общая документация к версии API, которая выше версии в вики (5.22 vs 5.35), то к ней надо относиться с настороженностью.**
**Однако, это может сильно помочь для дальнейшего понимания.**

Из-за отсутствия спецификации на конкретную имплементацию сервиса, неясно, какие атрибуты будут присутствовать всегда (т.е. они `required`), а какие - нет (`optional`).
По этой причине не удалось вытащить должность сотрудника для подразделения - у кого-то такая информация есть, а у кого-то ее нет!

### Проблема ручного парсинга

Ручной парсинг json'ов - неприятное занятие.
Если хочется и дальше получать ответы с сервера, то нужно точно определить, какие поля нам нужно оставлять, а от каких отказываться.

Исходя из этого понимания, можно будет использовать методы, предоставляемые различными библиотеками.
Возможно, это позволит создать более fault tolerant решение.

### Что делать дальше?

Пока ручное построение запросов в pandas работает, оно не очень удобно для дальнейшей работы - держать несколько pandas dataframes одновременно в ram может быть проблематичным.
Возможно, стоит перейти к связке "бд + сервис запроса"

В целом, реляционная бд может подойти, но требуется понять, как организовать схему БД для укладывания существующих сущностей.
Исходя из увиденнного, один человек может принадлежать нескольким организационным подразделениям.
А организационные подразделения выстраивают иерархию.

Надо более подробно изучить ВСЕ сущности и понять связи между ними (напр. найти или сформировать ERM-диаграмму), определить, от каких атрибутов можно отказаться.

Наверное, это позволить сделать сущности более плоскими и удобными для укладки в БД и дальнейшего исследования данных.

## Источники для дальнейшего изучения

### Интернет-ресурсы

Список ресурсов, которые можно использовать для дальнейшей работы:

- [Pure Help Center: Documentation](https://helpcenter.pure.elsevier.com/en_US/documentation)
- [Pure API User Guide](https://helpcenter.pure.elsevier.com/pure-api-home)
- [Understanding Dependents in the Pure API](https://helpcenter.pure.elsevier.com/understanding-dependents-in-the-api)
- [Differences Between UUID and Pure ID](https://helpcenter.pure.elsevier.com/difference-between-uuid-and-pure-id)
- [Swagger: Pure API Specification](https://api.elsevierpure.com/ws/api/api-docs/index.html?url=/ws/api/openapi.yaml)
- [Overview of Content Types](https://helpcenter.pure.elsevier.com/overview-of-content-types)
- [Organisational Unit](https://helpcenter.pure.elsevier.com/organisational-unit)

### Литература и общение с AI по теме задачи

#### Pure от Elseveir - это CRIS / RIMS

Некоторые заметки по тому, что такое Pure в принципе.
Это будет полезно для поиска статей, посвященных таким системам.

Согласно [Википедии](https://en.wikipedia.org/wiki/Current_research_information_system):

> CRIS — это база данных или иная информационная система для хранения, управления и обмена контекстными метаданными об исследовательской деятельности, финансируемой исследовательским фондом или проводимой в организации, выполняющей исследования (или их объединении).

CRIS — это синоним RIMS (Research Information Management System, Система управления исследовательской информацией).
Pure — одна из таких систем.

### BI для RIMS

Википедия также дает краткую информацию о стороне Business Intelligence для RIMS:

> Благодаря комплексной агрегации контекстной исследовательской информации CRIS являются очень подходящими инструментами для извлечения показателей бизнес-аналитики для принятия решений в учреждениях и за их пределами.

Таким образом, мы можем попытаться найти статьи о принятии решений с использованием RIMS/CRIS.
Кроме того, я полагаю, что [SciVal](https://www.scival.com/landing) от Elsevier существует именно для этой цели.

Нашла немного литературы по этой теме:

- Статья Никифоровой с элементами предиктивного анализа (машинное обучение и статистика)
- Нашла обзорный PDF по Scival; предлагает несколько use cases для применения метрик

SciVal похож на то, что хотелось бы получить на выходе.
Вот что Gemini говорит о Scival:

> SciVal — это веб-аналитическое решение от Elsevier, использующее данные Scopus для визуализации, анализа и бенчмаркинга исследовательской эффективности более 20 000 учреждений, 10 000+ исследовательских тем и связанных с ними исследователей из более чем 230 стран.
>
> Оно обеспечивает стратегическое принятие решений, выявление партнеров и отслеживание тенденций, например, в области ЦУР (SDGs) и научных направлениях.
>
> Ключевые особенности и возможности:
>
> - **Бенчмаркинг (Сравнительный анализ):** Сравнивайте исследовательскую эффективность (публикационная активность, цитируемость, влияние) учреждений, команд или отдельных лиц с конкурентами, используя такие показатели, как FWCI (Field-Weighted Citation Impact — взвешенный по области науки показатель цитирования).
> - **Сотрудничество:** Определяйте существующих или потенциальных партнеров путем анализа сетей соавторства и поиска ведущих экспертов в конкретных областях.
> - **Анализ трендов:** Изучайте исследовательские тренды, тематические кластеры и актуальные темы для выявления новых областей исследований.
> - **Отчетность:** Создавайте настраиваемые отчеты для демонстрации научного влияния (research impact) в целях получения финансирования, найма сотрудников или участия в рейтингах.
> - **Источники данных:** Использует данные Scopus, охватывающие период с 1996 года по настоящее время и включающие более 80 миллионов записей от 7000+ издателей.

Итак... еще одна тема для ресерча: что такое бенчмаркинг и какие метрики существуют в области CRIS и академической среды.